# Disclosure: Funds Lending and Endorsement/Guarantee

Track related-party transactions: funds lending, endorsements, and guarantees from MOPS.

In [4]:
from twmops import DisclosureFetcher
import pandas as pd

## Fetch Disclosure Data

Get disclosure data once - then analyze it multiple ways.

In [5]:
fetcher = DisclosureFetcher()

try:
    # Fetch TSMC (2330) disclosure data
    result = fetcher.get_disclosure("2330", year=115, month=3)
except Exception as e:
    print(f"Error fetching disclosure data: {e}")
    print("Note: This may be a temporary MOPS server issue. Please try again later.")
    result = None

if result:
    print(f"{result.company_name} ({result.stock_id})")
    print(f"Report date: {result.year}/{result.month}")
    print()

SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi


台積電 (2330)
Report date: 115/3



## Funds Lending Records

## Field Reference

**FundsLending:**
- `entity`: Entity name
- `has_balance`: Whether there's an outstanding balance
- `current_month`: Lending amount in current month (thousands TWD)
- `previous_month`: Lending amount in previous month (thousands TWD)
- `max_limit`: Maximum limit (thousands TWD)

**EndorsementGuarantee:**
- `entity`: Entity name
- `has_balance`: Whether there's an outstanding balance
- `monthly_change`: Change in current month (thousands TWD)
- `accumulated_balance`: Total accumulated balance (thousands TWD)
- `max_limit`: Maximum limit (thousands TWD)

**CrossCompanyGuarantee:**
- `parent_to_subsidiary`: Parent company guarantee to subsidiary (thousands TWD)
- `subsidiary_to_parent`: Subsidiary guarantee to parent (thousands TWD)

**ChinaGuarantee:**
- `entity`: Entity name
- `has_balance`: Whether there's an outstanding balance
- `monthly_change`: Change in current month (thousands TWD)
- `accumulated_balance`: Total accumulated balance (thousands TWD)

In [6]:
if result:
    print(f"Related-Party Transaction Summary for {result.company_name}")
    print(f"Report date: {result.year}/{result.month}")
    print()
    
    # Funds lending summary
    if result.funds_lending:
        total_current = sum(l.current_month or 0 for l in result.funds_lending)
        print(f"Funds Lending:")
        print(f"  Total records: {len(result.funds_lending)}")
        print(f"  Total current month: NT${total_current:,} thousand")
        print()
    
    # Endorsement/guarantee summary
    if result.endorsement_guarantee:
        total_balance = sum(g.accumulated_balance or 0 for g in result.endorsement_guarantee)
        print(f"Endorsement/Guarantee:")
        print(f"  Total records: {len(result.endorsement_guarantee)}")
        print(f"  Total accumulated balance: NT${total_balance:,} thousand")
        print()
    
    # Cross-company guarantees
    if result.cross_company:
        print(f"Cross-Company Guarantees:")
        if result.cross_company.parent_to_subsidiary is not None:
            print(f"  Parent to subsidiary: NT${result.cross_company.parent_to_subsidiary:,} thousand")
        if result.cross_company.subsidiary_to_parent is not None:
            print(f"  Subsidiary to parent: NT${result.cross_company.subsidiary_to_parent:,} thousand")
        print()
    
    # China operations
    if result.china_guarantee:
        print(f"China Operations Guarantee:")
        print(f"  Total records: {len(result.china_guarantee)}")
        total_china = sum(c.accumulated_balance or 0 for c in result.china_guarantee)
        print(f"  Total accumulated balance: NT${total_china:,} thousand")
        for txn in result.china_guarantee[:3]:
            print(f"    {txn.entity}: NT${txn.accumulated_balance or 0:,} thousand")

Related-Party Transaction Summary for 台積電
Report date: 115/3

Funds Lending:
  Total records: 2
  Total current month: NT$14,991,840 thousand

Endorsement/Guarantee:
  Total records: 2
  Total accumulated balance: NT$691,753,511 thousand

Cross-Company Guarantees:
  Parent to subsidiary: NT$691,753,511 thousand
  Subsidiary to parent: NT$0 thousand

China Operations Guarantee:
  Total records: 2
  Total accumulated balance: NT$0 thousand
    本公司: NT$0 thousand
    子公司: NT$0 thousand


## Related-Party Transaction Summary

In [7]:
if result and result.endorsement_guarantee:
    print(f"Endorsement/Guarantee Records: {len(result.endorsement_guarantee)}")
    print()
    for guarantee in result.endorsement_guarantee[:5]:
        print(f"Entity: {guarantee.entity}")
        print(f"  Has balance: {guarantee.has_balance}")
        if guarantee.monthly_change is not None:
            print(f"  Monthly change: NT${guarantee.monthly_change:,} thousand")
        if guarantee.accumulated_balance is not None:
            print(f"  Accumulated balance: NT${guarantee.accumulated_balance:,} thousand")
        if guarantee.max_limit is not None:
            print(f"  Max limit: NT${guarantee.max_limit:,} thousand")
        print()
    if len(result.endorsement_guarantee) > 5:
        print(f"... and {len(result.endorsement_guarantee) - 5} more records")
else:
    print("No endorsement/guarantee records")

Endorsement/Guarantee Records: 2

Entity: 本公司
  Has balance: True
  Monthly change: NT$17,345,616 thousand
  Accumulated balance: NT$691,753,511 thousand
  Max limit: NT$2,167,838,398 thousand

Entity: 子公司
  Has balance: False
  Monthly change: NT$0 thousand
  Accumulated balance: NT$0 thousand
  Max limit: NT$0 thousand



## Endorsement and Guarantee Records

In [8]:
if result and result.funds_lending:
    print(f"Funds Lending Records: {len(result.funds_lending)}")
    print()
    for lending in result.funds_lending[:5]:
        print(f"Entity: {lending.entity}")
        print(f"  Has balance: {lending.has_balance}")
        if lending.current_month is not None:
            print(f"  Current month: NT${lending.current_month:,} thousand")
        if lending.previous_month is not None:
            print(f"  Previous month: NT${lending.previous_month:,} thousand")
        if lending.max_limit is not None:
            print(f"  Max limit: NT${lending.max_limit:,} thousand")
        print()
    if len(result.funds_lending) > 5:
        print(f"... and {len(result.funds_lending) - 5} more lending records")
else:
    print("No funds lending records")

Funds Lending Records: 2

Entity: 本公司
  Has balance: False
  Current month: NT$0 thousand
  Previous month: NT$0 thousand
  Max limit: NT$0 thousand

Entity: 子公司
  Has balance: True
  Current month: NT$14,991,840 thousand
  Previous month: NT$12,835,680 thousand
  Max limit: NT$161,235,966 thousand



## Async Version (Concurrent)

Use async/await for concurrent requests in Jupyter notebooks.

Alternatively, you can use the async version if you need concurrent requests or are in an async environment.

## Fetch Endorsement and Guarantee Records

Get endorsement/guarantee transactions.

In [11]:
async def fetch_endorsement_guarantee():
    fetcher = DisclosureFetcher()
    
    try:
        result = await fetcher.get_disclosure_async("2330", year=115, month=3)
    except Exception as e:
        print(f"Error fetching disclosure data: {e}")
        print("Note: This may be a temporary MOPS server issue. Please try again later.")
        return None
    
    print(f"Endorsement/Guarantee for {result.company_name}")
    print(f"Total records: {len(result.endorsement_guarantee)}")

endorsement = await fetch_endorsement_guarantee()

SSL verification failed for MOPS; retrying without verification. This may be due to MOPS certificate issues. Consider updating your SSL certificates with: pip install certifi


Endorsement/Guarantee for 台積電
Total records: 2


## Field Reference

**FundsLending:**
- `entity`: Entity name
- `has_balance`: Whether there's an outstanding balance
- `current_month`: Lending amount in current month (thousands TWD)
- `previous_month`: Lending amount in previous month (thousands TWD)
- `max_limit`: Maximum limit (thousands TWD)

**EndorsementGuarantee:**
- `entity`: Entity name
- `has_balance`: Whether there's an outstanding balance
- `monthly_change`: Change in current month (thousands TWD)
- `accumulated_balance`: Total accumulated balance (thousands TWD)
- `max_limit`: Maximum limit (thousands TWD)

**CrossCompanyGuarantee:**
- `parent_to_subsidiary`: Parent company guarantee to subsidiary (thousands TWD)
- `subsidiary_to_parent`: Subsidiary guarantee to parent (thousands TWD)

**ChinaGuarantee:**
- `entity`: Entity name
- `has_balance`: Whether there's an outstanding balance
- `monthly_change`: Change in current month (thousands TWD)
- `accumulated_balance`: Total accumulated balance (thousands TWD)